In [1]:
# Import PuLP modeler functions
from pulp import *

In [3]:
# Creates a list of all the supply nodes
Plants = ["Marietta", "Minneapolis"]

# Creates a dictionary for the number of units of supply for each supply node
supply = {"Marietta": 1200,
          "Minneapolis": 800}

# Creates a list of all demand nodes
DistCenters = ["Cleveland", "Baltimore", "Chicago", "Phoenix"]

# Creates a dictionary for the number of units of demand for each demand node
demand = {"Cleveland":150,
          "Baltimore":350,
          "Chicago":500,
          "Phoenix":1000}
# Creates a list of costs of each transportation path
costs = [[12.60,14.35,11.52,17.58],
         [9.75,16.26,8.11,17.92]
         ]
# The cost data is made into a dictionary
costs = makeDict([Plants,DistCenters],costs,0)

In [5]:
costs

defaultdict(<function pulp.utilities.__makeDict.<locals>.<lambda>()>,
            {'Marietta': defaultdict(<function pulp.utilities.__makeDict.<locals>.<lambda>()>,
                         {'Cleveland': 12.6,
                          'Baltimore': 14.35,
                          'Chicago': 11.52,
                          'Phoenix': 17.58}),
             'Minneapolis': defaultdict(<function pulp.utilities.__makeDict.<locals>.<lambda>()>,
                         {'Cleveland': 9.75,
                          'Baltimore': 16.26,
                          'Chicago': 8.11,
                          'Phoenix': 17.92})})

In [7]:
# Creates the 'prob' variable to contain the problem data
prob = LpProblem("Transportation_Problem",LpMinimize)

In [9]:
# A dictionary called 'Vars' is created to contain the referenced variables(the routes)
vars = LpVariable.dicts("Route",(Plants,DistCenters),0)

In [11]:
vars

{'Marietta': {'Cleveland': Route_Marietta_Cleveland,
  'Baltimore': Route_Marietta_Baltimore,
  'Chicago': Route_Marietta_Chicago,
  'Phoenix': Route_Marietta_Phoenix},
 'Minneapolis': {'Cleveland': Route_Minneapolis_Cleveland,
  'Baltimore': Route_Minneapolis_Baltimore,
  'Chicago': Route_Minneapolis_Chicago,
  'Phoenix': Route_Minneapolis_Phoenix}}

In [13]:
Routes = [(p,d) for p in Plants for d in DistCenters]

In [15]:
Routes

[('Marietta', 'Cleveland'),
 ('Marietta', 'Baltimore'),
 ('Marietta', 'Chicago'),
 ('Marietta', 'Phoenix'),
 ('Minneapolis', 'Cleveland'),
 ('Minneapolis', 'Baltimore'),
 ('Minneapolis', 'Chicago'),
 ('Minneapolis', 'Phoenix')]

In [17]:
# The objective function is added to 'prob' first
prob += lpSum([vars[p][d]*costs[p][d] for (p,d) in Routes]), "Sum_of_Transporting_Costs"

In [19]:
# The supply maximum constraints are added to prob for each supply node (warehouse)
for p in Plants:
    prob += lpSum([vars[p][d] for d in DistCenters]) <= supply[p], "Sum_of_Products_out_of_Plant_%s" %p

# The demand minimum constraints are added to prob for each demand node (bar)
for d in DistCenters:
    prob += lpSum([vars[p][d] for p in Plants]) == demand[d], "Sum_of_Products_into_Distribution_Center_%s" %d

In [21]:
prob

Transportation_Problem:
MINIMIZE
14.35*Route_Marietta_Baltimore + 11.52*Route_Marietta_Chicago + 12.6*Route_Marietta_Cleveland + 17.58*Route_Marietta_Phoenix + 16.26*Route_Minneapolis_Baltimore + 8.11*Route_Minneapolis_Chicago + 9.75*Route_Minneapolis_Cleveland + 17.92*Route_Minneapolis_Phoenix + 0.0
SUBJECT TO
Sum_of_Products_out_of_Plant_Marietta: Route_Marietta_Baltimore
 + Route_Marietta_Chicago + Route_Marietta_Cleveland + Route_Marietta_Phoenix
 <= 1200

Sum_of_Products_out_of_Plant_Minneapolis: Route_Minneapolis_Baltimore
 + Route_Minneapolis_Chicago + Route_Minneapolis_Cleveland
 + Route_Minneapolis_Phoenix <= 800

Sum_of_Products_into_Distribution_Center_Cleveland: Route_Marietta_Cleveland
 + Route_Minneapolis_Cleveland = 150

Sum_of_Products_into_Distribution_Center_Baltimore: Route_Marietta_Baltimore
 + Route_Minneapolis_Baltimore = 350

Sum_of_Products_into_Distribution_Center_Chicago: Route_Marietta_Chicago
 + Route_Minneapolis_Chicago = 500

Sum_of_Products_into_Distribut

In [23]:
# The problem is solved using PuLP's choice of Solver
prob.solve()

1

In [25]:
# Each of the variables is printed with it's resolved optimum value
for v in prob.variables():
    print(v.name, "=", v.varValue)
# The optimised objective function value is printed to the screen
print("Total Cost of Transportation = ", value(prob.objective))

Route_Marietta_Baltimore = 350.0
Route_Marietta_Chicago = 0.0
Route_Marietta_Cleveland = 0.0
Route_Marietta_Phoenix = 850.0
Route_Minneapolis_Baltimore = 0.0
Route_Minneapolis_Chicago = 500.0
Route_Minneapolis_Cleveland = 150.0
Route_Minneapolis_Phoenix = 150.0
Total Cost of Transportation =  28171.0


In [27]:
import pandas as pd
print("\nSensitivity Analysis")
Cons_Sensitivity_Report=[{'Constraint_Name':name,'Slack':c.slack,"Shadow_Price":c.pi}
                         for name, c in prob.constraints.items()]
print(pd.DataFrame(Cons_Sensitivity_Report))
print("\n")
Variable_Sensitivity_Report = [{'Variable_Name': v.name, 'Value': v.varValue,'Reduced_Cost': v.dj}
                               for v in prob.variables()]

print(pd.DataFrame(Variable_Sensitivity_Report)[['Variable_Name','Value','Reduced_Cost']])


Sensitivity Analysis
                                     Constraint_Name  Slack  Shadow_Price
0              Sum_of_Products_out_of_Plant_Marietta   -0.0         -0.34
1           Sum_of_Products_out_of_Plant_Minneapolis   -0.0          0.00
2  Sum_of_Products_into_Distribution_Center_Cleve...   -0.0          9.75
3  Sum_of_Products_into_Distribution_Center_Balti...   -0.0         14.69
4   Sum_of_Products_into_Distribution_Center_Chicago   -0.0          8.11
5   Sum_of_Products_into_Distribution_Center_Phoenix   -0.0         17.92


                 Variable_Name  Value  Reduced_Cost
0     Route_Marietta_Baltimore  350.0          0.00
1       Route_Marietta_Chicago    0.0          3.75
2     Route_Marietta_Cleveland    0.0          3.19
3       Route_Marietta_Phoenix  850.0          0.00
4  Route_Minneapolis_Baltimore    0.0          1.57
5    Route_Minneapolis_Chicago  500.0          0.00
6  Route_Minneapolis_Cleveland  150.0          0.00
7    Route_Minneapolis_Phoenix  150.0     